# Vision QA — inventarul cadrelor și etichetele

**Scop.** Verifică ce cadre există, ce etichete au și cum se poate împărți datasetul fără
scurgere între antrenare și validare.

**Intrare.** `simulation/mujoco/runs/*/frames.jsonl` și PNG-urile din `frames/`.
**Ieșire.** `ml/artifacts/qa/qa-frames.json`, cu rândurile și splitul propus.

**Regulă.** `family`, `defect`, `verdict` și `is_synthetic_defect` sunt etichete. Nu intră
niciodată în intrarea modelului; cadrul de inspecție le exclude deja prin `LABEL_FIELDS`.


## 1. Mediu și importuri

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "ml" / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "ml" / "src").is_dir():
    raise RuntimeError("rulează notebook-ul din repository")
sys.path.insert(0, str(ROOT / "ml" / "src"))
print("repository:", ROOT)

from collections import Counter

from cncdt_ml.mujoco_corpus import frame_path, load_frames, session_directories


## 2. Configurare

Implicit se citesc toate sesiunile de sub `runs/`, fiindcă o singură rulare poate să nu aibă
cadre. Schimbă `RUN` pe o rulare anume când vrei un split reproductibil.


In [ ]:
RUNS = ROOT / "simulation" / "mujoco" / "runs"
RUN = RUNS
ARTIFACTS = ROOT / "ml" / "artifacts" / "qa"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("rulare:", RUN)


## 3. Inventarul cadrelor

In [ ]:
rows = []
for directory in session_directories(RUN):
    for frame in load_frames(directory):
        rows.append({"session": directory.name, **frame, "absolute_path": str(frame_path(directory, frame))})

print(len(rows), "cadre")
for field in ("verdict", "family", "defect", "pov", "stage"):
    counts = Counter(row.get(field, "") for row in rows)
    print(field, dict(sorted(counts.items())))


## 4. Verifică fișierele

In [ ]:
missing = [row["absolute_path"] for row in rows if not Path(row["absolute_path"]).is_file()]
print("imagini lipsă:", len(missing))
if missing:
    print("\n".join(missing[:10]))


## 5. Câte clase are fiecare sesiune

Un split aleator pe cadre ar pune aproape sigur aceeași piesă în antrenare și validare. Splitul
se face pe ciclu, iar când există mai multe sesiuni, pe sesiune.


In [ ]:
for session in sorted({row["session"] for row in rows}):
    subset = [row for row in rows if row["session"] == session]
    print(session, dict(Counter(row.get("verdict") for row in subset)))


## 6. Split pe cicluri

Splitul se face separat pe ciclurile cu `REJECT` și pe cele cu `PASS`, ca ambele seturi să
conțină ambele clase. Cu o singură sesiune care are defecte, un split pe sesiune ar lăsa
validarea fără `REJECT`.


In [ ]:
def split_group(cycles, val_fraction=0.4):
    ordered = sorted(cycles)
    cut = max(1, int(round(len(ordered) * val_fraction)))
    return set(ordered[:cut]), set(ordered[cut:])


cycles = sorted({(row["session"], row["cycle_id"]) for row in rows})
reject_cycles = sorted({(row["session"], row["cycle_id"]) for row in rows if row["verdict"] == "REJECT"})
pass_cycles = sorted(set(cycles) - set(reject_cycles))
val_reject, train_reject = split_group(reject_cycles)
val_pass, train_pass = split_group(pass_cycles)
train_cycles = train_reject | train_pass
val_cycles = val_reject | val_pass

train_rows = [row for row in rows if (row["session"], row["cycle_id"]) in train_cycles]
val_rows = [row for row in rows if (row["session"], row["cycle_id"]) in val_cycles]
print("train", len(train_rows), dict(Counter(row["verdict"] for row in train_rows)))
print("val", len(val_rows), dict(Counter(row["verdict"] for row in val_rows)))


## 7. Scrie manifestul

In [ ]:
manifest = {
    "run": str(RUN),
    "frames": rows,
    "train_cycles": sorted(train_cycles),
    "val_cycles": sorted(val_cycles),
}
manifest_path = ARTIFACTS / "qa-frames.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("scris", manifest_path)


## 8. Pasul următor

Corpusul actual are 35 de cadre `REJECT`, toate într-o singură sesiune (`demo-wheel_100-rate50`).
Pentru un clasificator care generalizează, următoarea colectare trebuie să adauge defecte în
mai multe familii și sesiuni. Notebook-ul de baseline este `qa-02-vision.ipynb`.
